In [1]:
# src/utils

import sys
sys.path.append("/home/jovyan/work")

from src.utils.spark_session import get_spark_session
spark = get_spark_session("generate-sample-data")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DataFrame[]

In [2]:
# Reference/ congig tables

from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, LongType, StringType, DateType
from datetime import datetime

now = datetime.now()

# --- branch ---
df_branch = spark.createDataFrame([
    Row(branch_id=1, name="Lagos - Head Office", address="Ikeja, Lagos",
        is_active=True, created_at=now, updated_at=now),
])
df_branch.write.format("delta").mode("overwrite").saveAsTable("silver.branch")

# --- data_source ---
df_data_source = spark.createDataFrame([
    Row(source_id=1, source_code="GOOGLE_FORM", description="Google Forms submission"),
    Row(source_id=2, source_code="WHATSAPP_GROUP_AGENT", description="WhatsApp group message, LLM-extracted"),
    Row(source_id=3, source_code="RECEIPT_AGENT", description="WhatsApp receipt photo, OCR-extracted"),
    Row(source_id=4, source_code="MANUAL", description="Manual entry / correction"),
])
df_data_source.write.format("delta").mode("overwrite").saveAsTable("silver.data_source")

# --- auction_house ---
df_auction_house = spark.createDataFrame([
    Row(auction_house_id=1, name="Copart", country="USA", created_at=now),
    Row(auction_house_id=2, name="IAAI", country="USA", created_at=now),
    Row(auction_house_id=3, name="Manheim", country="USA", created_at=now),
])
df_auction_house.write.format("delta").mode("overwrite").saveAsTable("silver.auction_house")

# --- vendor ---
df_vendor = spark.createDataFrame([
    Row(vendor_id=1, name="AutoParts Lagos Ltd", contact_phone="+2348012345001", contact_email="sales@autopartslagos.ng", created_at=now),
    Row(vendor_id=2, name="Naija Spares Hub", contact_phone="+2348012345002", contact_email="info@naijaspares.ng", created_at=now),
    Row(vendor_id=3, name="Trust Motor Spares", contact_phone="+2348012345003", contact_email=None, created_at=now),
])
df_vendor.write.format("delta").mode("overwrite").saveAsTable("silver.vendor")

# --- part ---
df_part = spark.createDataFrame([
    Row(part_id=1, name="Front Bumper", category="Body", created_at=now),
    Row(part_id=2, name="Headlight Assembly", category="Body", created_at=now),
    Row(part_id=3, name="Side Mirror", category="Body", created_at=now),
    Row(part_id=4, name="Windshield", category="Body", created_at=now),
    Row(part_id=5, name="Brake Pads", category="Mechanical", created_at=now),
    Row(part_id=6, name="Alternator", category="Electrical", created_at=now),
    Row(part_id=7, name="Radiator", category="Mechanical", created_at=now),
    Row(part_id=8, name="Seat Cover", category="Interior", created_at=now),
])
df_part.write.format("delta").mode("overwrite").saveAsTable("silver.part")

# --- staff ---
df_staff = spark.createDataFrame([
    Row(staff_id=1, branch_id=1, name="Tunde Bakare", phone_whatsapp="+2348030000001", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=2, branch_id=1, name="Chinedu Okafor", phone_whatsapp="+2348030000002", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=3, branch_id=1, name="Musa Ibrahim", phone_whatsapp="+2348030000003", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=4, branch_id=1, name="Ifeoma Nwosu", phone_whatsapp="+2348030000004", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=5, branch_id=1, name="Segun Adeyemi", phone_whatsapp="+2348030000005", is_active=True, created_at=now, updated_at=now),
    Row(staff_id=6, branch_id=1, name="Amaka Eze", phone_whatsapp="+2348030000006", is_active=True, created_at=now, updated_at=now),
])
df_staff.write.format("delta").mode("overwrite").saveAsTable("silver.staff")


# --- staff_role (who does what — matches "roles not fixed headcount" from Phase 1) ---
staff_role_schema = StructType([
    StructField("staff_id", LongType(), False),
    StructField("role_code", StringType(), False),
    StructField("valid_from", DateType(), False),
    StructField("valid_to", DateType(), True),
])

df_staff_role = spark.createDataFrame([
    Row(staff_id=1, role_code="OWNER", valid_from=now.date(), valid_to=None),
    Row(staff_id=1, role_code="FINANCE", valid_from=now.date(), valid_to=None),
    Row(staff_id=2, role_code="DRIVER", valid_from=now.date(), valid_to=None),
    Row(staff_id=3, role_code="DRIVER", valid_from=now.date(), valid_to=None),
    Row(staff_id=4, role_code="INSPECTOR", valid_from=now.date(), valid_to=None),
    Row(staff_id=4, role_code="MECHANIC", valid_from=now.date(), valid_to=None),
    Row(staff_id=5, role_code="MECHANIC", valid_from=now.date(), valid_to=None),
    Row(staff_id=6, role_code="SALES", valid_from=now.date(), valid_to=None),
], schema=staff_role_schema)
df_staff_role.write.format("delta").mode("overwrite").saveAsTable("silver.staff_role")


# --- inspection_checklist_item ---
df_checklist = spark.createDataFrame([
    Row(item_id=1, category="ENGINE", label="Engine starts and runs smoothly", is_active=True, sort_order=1),
    Row(item_id=2, category="ENGINE", label="No visible oil/coolant leaks", is_active=True, sort_order=2),
    Row(item_id=3, category="BODY", label="No major dents or rust", is_active=True, sort_order=3),
    Row(item_id=4, category="BODY", label="Panel alignment is correct", is_active=True, sort_order=4),
    Row(item_id=5, category="ELECTRICAL", label="All lights functional", is_active=True, sort_order=5),
    Row(item_id=6, category="ELECTRICAL", label="AC/heating works", is_active=True, sort_order=6),
    Row(item_id=7, category="INTERIOR", label="Seats and upholstery in good condition", is_active=True, sort_order=7),
    Row(item_id=8, category="TIRES", label="Tire tread depth acceptable", is_active=True, sort_order=8),
])
df_checklist.write.format("delta").mode("overwrite").saveAsTable("silver.inspection_checklist_item")

for t in ["branch", "data_source", "auction_house", "vendor", "part", "staff", "staff_role", "inspection_checklist_item"]:
    print(f"silver.{t}: {spark.table(f'silver.{t}').count()} rows")

silver.branch: 1 rows
silver.data_source: 4 rows
silver.auction_house: 3 rows
silver.vendor: 3 rows
silver.part: 8 rows
silver.staff: 6 rows
silver.staff_role: 8 rows
silver.inspection_checklist_item: 8 rows


In [3]:
from faker import Faker
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType, IntegerType,
    DateType, TimestampType, BooleanType, DecimalType
)
from decimal import Decimal
from datetime import timedelta
import random

fake = Faker()
Faker.seed(42)
random.seed(42)

MAKE_MODELS = [
    ("Toyota", "Camry"), ("Toyota", "Corolla"), ("Toyota", "Highlander"), ("Toyota", "RAV4"),
    ("Honda", "Accord"), ("Honda", "CR-V"), ("Honda", "Civic"),
    ("Lexus", "RX350"), ("Lexus", "ES350"),
    ("Ford", "Edge"), ("Ford", "Explorer"),
    ("Nissan", "Altima"), ("Nissan", "Murano"),
]

N_VEHICLES = 40
today = datetime.now()

vehicle_schema = StructType([
    StructField("vehicle_id", LongType(), False),
    StructField("vin", StringType(), True),
    StructField("auction_lot_no", StringType(), True),
    StructField("make", StringType(), True),
    StructField("model", StringType(), True),
    StructField("model_year", IntegerType(), True),
    StructField("branch_id", LongType(), False),
    StructField("current_stage", StringType(), False),
    StructField("current_status", StringType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

purchase_schema = StructType([
    StructField("purchase_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("auction_house_id", LongType(), True),
    StructField("buyer_staff_id", LongType(), True),
    StructField("price_amount", DecimalType(14, 2), False),
    StructField("price_currency", StringType(), False),
    StructField("purchase_date", DateType(), False),
    StructField("status", StringType(), False),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

vehicle_rows, purchase_rows = [], []

for i in range(1, N_VEHICLES + 1):
    make, model = random.choice(MAKE_MODELS)
    purchase_date = today - timedelta(days=random.randint(1, 180))
    price = Decimal(str(round(random.uniform(1500, 9000), 2)))

    vehicle_rows.append(Row(
        vehicle_id=i,
        vin=fake.bothify(text="?????????????????").upper(),
        auction_lot_no=fake.bothify(text="LOT-#####"),
        make=make, model=model,
        model_year=random.randint(2012, 2020),
        branch_id=1,
        current_stage="PURCHASED",
        current_status="ACTIVE",
        created_at=purchase_date, updated_at=purchase_date,
        is_deleted=False, deleted_at=None,
    ))

    purchase_rows.append(Row(
        purchase_id=i, vehicle_id=i,
        auction_house_id=random.randint(1, 3),
        buyer_staff_id=1,
        price_amount=price,
        price_currency="USD",
        purchase_date=purchase_date.date(),
        status="CONFIRMED",
        source_id=1,
        source_record_id=f"FORM-PUR-{i:04d}",
        created_at=purchase_date, updated_at=purchase_date,
        is_deleted=False, deleted_at=None,
    ))

df_vehicle = spark.createDataFrame(vehicle_rows, schema=vehicle_schema)
df_vehicle.write.format("delta").mode("overwrite").saveAsTable("silver.vehicle")

df_purchase = spark.createDataFrame(purchase_rows, schema=purchase_schema)
df_purchase.write.format("delta").mode("overwrite").saveAsTable("silver.purchase")

print(f"silver.vehicle: {spark.table('silver.vehicle').count()} rows")
print(f"silver.purchase: {spark.table('silver.purchase').count()} rows")
spark.table("silver.vehicle").show(5)

silver.vehicle: 40 rows
silver.purchase: 40 rows
+----------+-----------------+--------------+------+------+----------+---------+-------------+--------------+--------------------+--------------------+----------+----------+
|vehicle_id|              vin|auction_lot_no|  make| model|model_year|branch_id|current_stage|current_status|          created_at|          updated_at|is_deleted|deleted_at|
+----------+-----------------+--------------+------+------+----------+---------+-------------+--------------+--------------------+--------------------+----------+----------+
|         6|CZUZRENKTUNPFZPDJ|     LOT-42388| Lexus| RX350|      2012|        1|    PURCHASED|        ACTIVE|2026-02-17 12:34:...|2026-02-17 12:34:...|     false|      NULL|
|         7|QVLBLZXOIGFFWDHJO|     LOT-26916|Nissan|Altima|      2014|        1|    PURCHASED|        ACTIVE|2026-03-31 12:34:...|2026-03-31 12:34:...|     false|      NULL|
|         8|YMDHQJARUHRIWRXPV|     LOT-14627|Nissan|Murano|      2018|        1| 

In [4]:
purchases = spark.table("silver.purchase").select("vehicle_id", "purchase_date").collect()
today_date = today.date()

shipment_schema = StructType([
    StructField("shipment_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("carrier", StringType(), True),
    StructField("bill_of_lading_no", StringType(), True),
    StructField("etd", DateType(), True),
    StructField("eta", DateType(), True),
    StructField("ata", DateType(), True),
    StructField("status", StringType(), False),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

CARRIERS = ["Maersk Line", "MSC", "CMA CGM", "Hapag-Lloyd", "COSCO Shipping"]

shipment_rows = []
shipment_id = 1

for row in purchases:
    purchase_date = row.purchase_date
    elapsed_days = (today_date - purchase_date).days

    if elapsed_days < 2:
        continue  # too soon to have been booked/shipped yet

    etd = purchase_date + timedelta(days=2)
    shipping_duration = random.randint(20, 45)  # matches the 20-45 day SLA from Phase 1
    eta = etd + timedelta(days=shipping_duration)
    ata = eta if elapsed_days >= (2 + shipping_duration) else None
    status = "ARRIVED" if ata else "IN_TRANSIT"

    shipment_rows.append(Row(
        shipment_id=shipment_id,
        vehicle_id=row.vehicle_id,
        carrier=random.choice(CARRIERS),
        bill_of_lading_no=fake.bothify(text="BL########"),
        etd=etd, eta=eta, ata=ata,
        status=status,
        source_id=1,
        source_record_id=f"FORM-SHIP-{shipment_id:04d}",
        created_at=datetime.combine(etd, datetime.min.time()),
        updated_at=today,
        is_deleted=False, deleted_at=None,
    ))
    shipment_id += 1

df_shipment = spark.createDataFrame(shipment_rows, schema=shipment_schema)
df_shipment.write.format("delta").mode("overwrite").saveAsTable("silver.shipment")

print(f"silver.shipment: {spark.table('silver.shipment').count()} rows (out of {len(purchases)} vehicles)")
df_shipment.groupBy("status").count().show()

silver.shipment: 40 rows (out of 40 vehicles)
+----------+-----+
|    status|count|
+----------+-----+
|IN_TRANSIT|    4|
|   ARRIVED|   36|
+----------+-----+



In [5]:
shipments = spark.table("silver.shipment").filter("status = 'ARRIVED'").select("vehicle_id", "ata").collect()

port_arrival_schema = StructType([
    StructField("port_arrival_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("arrival_date", DateType(), False),
    StructField("clearance_status", StringType(), False),
    StructField("cleared_date", DateType(), True),
    StructField("hold_reason", StringType(), True),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

HOLD_REASONS = [
    "Awaiting duty payment confirmation",
    "Missing bill of lading endorsement",
    "Customs valuation dispute",
    "Documentation mismatch - VIN discrepancy",
]

port_arrival_rows = []
pid = 1

for row in shipments:
    arrival_date = row.ata
    elapsed = (today_date - arrival_date).days
    is_delayed = random.random() < 0.15
    normal_duration = random.randint(3, 14)
    extra_delay = random.randint(10, 30) if is_delayed else 0
    total_duration = normal_duration + extra_delay

    if elapsed >= total_duration:
        clearance_status = "CLEARED"
        cleared_date = arrival_date + timedelta(days=total_duration)
        hold_reason = random.choice(HOLD_REASONS) if is_delayed else None
    elif is_delayed:
        clearance_status = "ON_HOLD"
        cleared_date = None
        hold_reason = random.choice(HOLD_REASONS)
    else:
        clearance_status = "PENDING"
        cleared_date = None
        hold_reason = None

    port_arrival_rows.append(Row(
        port_arrival_id=pid,
        vehicle_id=row.vehicle_id,
        arrival_date=arrival_date,
        clearance_status=clearance_status,
        cleared_date=cleared_date,
        hold_reason=hold_reason,
        source_id=1,
        source_record_id=f"FORM-PORT-{pid:04d}",
        created_at=datetime.combine(arrival_date, datetime.min.time()),
        updated_at=today,
        is_deleted=False, deleted_at=None,
    ))
    pid += 1

df_port_arrival = spark.createDataFrame(port_arrival_rows, schema=port_arrival_schema)
df_port_arrival.write.format("delta").mode("overwrite").saveAsTable("silver.port_arrival")

print(f"silver.port_arrival: {spark.table('silver.port_arrival').count()} rows")
df_port_arrival.groupBy("clearance_status").count().show()

silver.port_arrival: 36 rows
+----------------+-----+
|clearance_status|count|
+----------------+-----+
|         CLEARED|   32|
|         PENDING|    3|
|         ON_HOLD|    1|
+----------------+-----+



In [6]:
cleared = spark.table("silver.port_arrival").filter("clearance_status = 'CLEARED'").select("vehicle_id", "cleared_date").collect()

pickup_schema = StructType([
    StructField("pickup_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("driver_staff_id", LongType(), True),
    StructField("pickup_date", DateType(), False),
    StructField("odometer", IntegerType(), True),
    StructField("photo_url", StringType(), True),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

office_intake_schema = StructType([
    StructField("intake_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("intake_date", DateType(), False),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

DRIVER_STAFF_IDS = [2, 3]  # from staff_role: role_code = 'DRIVER'

pickup_rows, intake_rows = [], []
pid, iid = 1, 1

for row in cleared:
    elapsed = (today_date - row.cleared_date).days
    if elapsed < 1:
        continue

    pickup_date = row.cleared_date + timedelta(days=1)

    pickup_rows.append(Row(
        pickup_id=pid,
        vehicle_id=row.vehicle_id,
        driver_staff_id=random.choice(DRIVER_STAFF_IDS),
        pickup_date=pickup_date,
        odometer=random.randint(35000, 130000),
        photo_url=f"https://placeholder.local/pickup/{row.vehicle_id}.jpg",
        source_id=1,
        source_record_id=f"FORM-PICKUP-{pid:04d}",
        created_at=datetime.combine(pickup_date, datetime.min.time()),
        updated_at=today,
        is_deleted=False, deleted_at=None,
    ))

    intake_rows.append(Row(
        intake_id=iid,
        vehicle_id=row.vehicle_id,
        intake_date=pickup_date,
        source_id=1,
        source_record_id=f"FORM-INTAKE-{iid:04d}",
        created_at=datetime.combine(pickup_date, datetime.min.time()),
        updated_at=today,
        is_deleted=False, deleted_at=None,
    ))
    pid += 1
    iid += 1

df_pickup = spark.createDataFrame(pickup_rows, schema=pickup_schema)
df_pickup.write.format("delta").mode("overwrite").saveAsTable("silver.pickup")

df_intake = spark.createDataFrame(intake_rows, schema=office_intake_schema)
df_intake.write.format("delta").mode("overwrite").saveAsTable("silver.office_intake")

print(f"silver.pickup: {spark.table('silver.pickup').count()} rows")
print(f"silver.office_intake: {spark.table('silver.office_intake').count()} rows")

silver.pickup: 32 rows
silver.office_intake: 32 rows


In [7]:
intakes = spark.table("silver.office_intake").select("vehicle_id", "intake_date").collect()

inspection_schema = StructType([
    StructField("inspection_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("inspector_staff_id", LongType(), True),
    StructField("inspection_date", DateType(), False),
    StructField("overall_result", StringType(), False),
    StructField("round_no", IntegerType(), False),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

inspection_result_schema = StructType([
    StructField("result_id", LongType(), False),
    StructField("inspection_id", LongType(), False),
    StructField("item_id", LongType(), False),
    StructField("pass_fail", StringType(), False),
    StructField("notes", StringType(), True),
])

damage_item_schema = StructType([
    StructField("damage_id", LongType(), False),
    StructField("inspection_id", LongType(), False),
    StructField("part_id", LongType(), True),
    StructField("severity", StringType(), False),
    StructField("description", StringType(), True),
    StructField("created_at", TimestampType(), False),
])

CHECKLIST_ITEM_IDS = [1, 2, 3, 4, 5, 6, 7, 8]
# maps checklist item -> plausible part(s) from our catalog; None where our
# 8-part catalog just doesn't cover it (e.g. no core-engine or tire part)
CHECKLIST_PART_MAP = {
    1: [None], 2: [7], 3: [1, 3, 4], 4: [1, 3],
    5: [2], 6: [None], 7: [8], 8: [None],
}
SEVERITIES = ["LOW", "MEDIUM", "HIGH"]

inspection_rows, result_rows, damage_rows = [], [], []
insp_id, result_id, damage_id = 1, 1, 1

for row in intakes:
    elapsed = (today_date - row.intake_date).days
    if elapsed < 1:
        continue

    inspection_date = row.intake_date + timedelta(days=1)
    overall_result = "PASS" if random.random() < 0.25 else "FAIL"
    n_fails = 0 if overall_result == "PASS" else random.randint(1, 3)
    failing_items = set(random.sample(CHECKLIST_ITEM_IDS, n_fails)) if n_fails else set()

    inspection_rows.append(Row(
        inspection_id=insp_id,
        vehicle_id=row.vehicle_id,
        inspector_staff_id=4,  # Ifeoma Nwosu, role INSPECTOR
        inspection_date=inspection_date,
        overall_result=overall_result,
        round_no=1,
        source_id=1,
        source_record_id=f"FORM-INSPECT-{insp_id:04d}",
        created_at=datetime.combine(inspection_date, datetime.min.time()),
        updated_at=today,
        is_deleted=False, deleted_at=None,
    ))

    for item_id in CHECKLIST_ITEM_IDS:
        pass_fail = "FAIL" if item_id in failing_items else "PASS"
        result_rows.append(Row(
            result_id=result_id, inspection_id=insp_id,
            item_id=item_id, pass_fail=pass_fail, notes=None,
        ))
        result_id += 1

        if pass_fail == "FAIL":
            damage_rows.append(Row(
                damage_id=damage_id,
                inspection_id=insp_id,
                part_id=random.choice(CHECKLIST_PART_MAP[item_id]),
                severity=random.choice(SEVERITIES),
                description=f"Flagged during inspection: checklist item {item_id} failed",
                created_at=datetime.combine(inspection_date, datetime.min.time()),
            ))
            damage_id += 1

    insp_id += 1

df_inspection = spark.createDataFrame(inspection_rows, schema=inspection_schema)
df_inspection.write.format("delta").mode("overwrite").saveAsTable("silver.inspection")

df_inspection_result = spark.createDataFrame(result_rows, schema=inspection_result_schema)
df_inspection_result.write.format("delta").mode("overwrite").saveAsTable("silver.inspection_result")

df_damage_item = spark.createDataFrame(damage_rows, schema=damage_item_schema)
df_damage_item.write.format("delta").mode("overwrite").saveAsTable("silver.damage_item")

print(f"silver.inspection: {spark.table('silver.inspection').count()} rows")
df_inspection.groupBy("overall_result").count().show()
print(f"silver.inspection_result: {spark.table('silver.inspection_result').count()} rows")
print(f"silver.damage_item: {spark.table('silver.damage_item').count()} rows")

silver.inspection: 32 rows
+--------------+-----+
|overall_result|count|
+--------------+-----+
|          FAIL|   24|
|          PASS|    8|
+--------------+-----+

silver.inspection_result: 256 rows
silver.damage_item: 54 rows


In [8]:
failed = spark.table("silver.inspection").filter("overall_result = 'FAIL' AND round_no = 1") \
    .select("inspection_id", "vehicle_id", "inspection_date").collect()
damage_items = spark.table("silver.damage_item").select("damage_id", "inspection_id", "part_id", "severity").collect()

from collections import defaultdict
damage_map = defaultdict(list)
for d in damage_items:
    damage_map[d.inspection_id].append(d)

repair_job_schema = StructType([
    StructField("repair_job_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("mechanic_staff_id", LongType(), True),
    StructField("start_date", DateType(), True),
    StructField("end_date", DateType(), True),
    StructField("status", StringType(), False),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

repair_job_damage_schema = StructType([
    StructField("repair_job_id", LongType(), False),
    StructField("damage_id", LongType(), False),
])

repair_job_part_schema = StructType([
    StructField("line_id", LongType(), False),
    StructField("repair_job_id", LongType(), False),
    StructField("part_id", LongType(), True),
    StructField("vendor_id", LongType(), True),
    StructField("qty", IntegerType(), False),
    StructField("unit_cost", DecimalType(12, 2), False),
    StructField("currency", StringType(), False),
    StructField("created_at", TimestampType(), False),
])

MECHANIC_STAFF_IDS = [4, 5]
VENDOR_IDS = [1, 2, 3]
SEVERITY_COST_NGN = {"LOW": (15000, 40000), "MEDIUM": (40000, 120000), "HIGH": (120000, 350000)}

repair_job_rows, repair_job_damage_rows, repair_job_part_rows = [], [], []
rj_id, line_id = 1, 1

for row in failed:
    elapsed_since_inspection = (today_date - row.inspection_date).days
    if elapsed_since_inspection < 1:
        continue

    start_date = row.inspection_date + timedelta(days=1)
    duration = random.randint(3, 14)
    elapsed_since_start = (today_date - start_date).days
    is_blocked = random.random() < 0.10

    if elapsed_since_start >= duration and not is_blocked:
        status, end_date = "DONE", start_date + timedelta(days=duration)
    elif is_blocked and elapsed_since_start < duration + 15:
        status, end_date = "BLOCKED_PARTS", None
    elif elapsed_since_start >= duration * 0.4:
        status, end_date = "IN_PROGRESS", None
    else:
        status, end_date = "OPEN", None

    repair_job_rows.append(Row(
        repair_job_id=rj_id, vehicle_id=row.vehicle_id,
        mechanic_staff_id=random.choice(MECHANIC_STAFF_IDS),
        start_date=start_date, end_date=end_date, status=status,
        source_id=1, source_record_id=f"FORM-REPAIR-{rj_id:04d}",
        created_at=datetime.combine(start_date, datetime.min.time()),
        updated_at=today, is_deleted=False, deleted_at=None,
    ))

    for d in damage_map[row.inspection_id]:
        repair_job_damage_rows.append(Row(repair_job_id=rj_id, damage_id=d.damage_id))
        if d.part_id is not None:
            lo, hi = SEVERITY_COST_NGN.get(d.severity, (20000, 80000))
            repair_job_part_rows.append(Row(
                line_id=line_id, repair_job_id=rj_id, part_id=d.part_id,
                vendor_id=random.choice(VENDOR_IDS), qty=1,
                unit_cost=Decimal(str(random.randint(lo, hi))), currency="NGN",
                created_at=datetime.combine(start_date, datetime.min.time()),
            ))
            line_id += 1

    rj_id += 1

df_repair_job = spark.createDataFrame(repair_job_rows, schema=repair_job_schema)
df_repair_job.write.format("delta").mode("overwrite").saveAsTable("silver.repair_job")

df_repair_job_damage = spark.createDataFrame(repair_job_damage_rows, schema=repair_job_damage_schema)
df_repair_job_damage.write.format("delta").mode("overwrite").saveAsTable("silver.repair_job_damage")

df_repair_job_part = spark.createDataFrame(repair_job_part_rows, schema=repair_job_part_schema)
df_repair_job_part.write.format("delta").mode("overwrite").saveAsTable("silver.repair_job_part")

print(f"silver.repair_job: {spark.table('silver.repair_job').count()} rows")
df_repair_job.groupBy("status").count().show() 
print(f"silver.repair_job_damage: {spark.table('silver.repair_job_damage').count()} rows")
print(f"silver.repair_job_part: {spark.table('silver.repair_job_part').count()} rows")

silver.repair_job: 23 rows
+-----------+-----+
|     status|count|
+-----------+-----+
|       DONE|   20|
|IN_PROGRESS|    3|
+-----------+-----+

silver.repair_job_damage: 52 rows
silver.repair_job_part: 35 rows


In [9]:
random.seed(42)

done_repairs = spark.table("silver.repair_job").filter("status = 'DONE'") \
    .select("vehicle_id", "end_date", "repair_job_id").collect()

reinspect_rows, reinspect_result_rows = [], []
insp_id2, result_id2 = 1000, 10000

for row in done_repairs:
    elapsed = (today_date - row.end_date).days
    if elapsed < 1:
        continue

    inspection_date = row.end_date + timedelta(days=1)
    overall_result = "PASS" if random.random() < 0.90 else "FAIL"

    reinspect_rows.append(Row(
        inspection_id=insp_id2, vehicle_id=row.vehicle_id,
        inspector_staff_id=4, inspection_date=inspection_date,
        overall_result=overall_result, round_no=2,
        source_id=1, source_record_id=f"FORM-INSPECT-{insp_id2:04d}",
        created_at=datetime.combine(inspection_date, datetime.min.time()),
        updated_at=today, is_deleted=False, deleted_at=None,
    ))

    for item_id in CHECKLIST_ITEM_IDS:
        pass_fail = "PASS" if overall_result == "PASS" else random.choice(["PASS", "PASS", "FAIL"])
        reinspect_result_rows.append(Row(
            result_id=result_id2, inspection_id=insp_id2, item_id=item_id,
            pass_fail=pass_fail, notes="Re-inspection after repair",
        ))
        result_id2 += 1

    insp_id2 += 1

df_reinspect = spark.createDataFrame(reinspect_rows, schema=inspection_schema)
df_reinspect.write.format("delta").mode("append").saveAsTable("silver.inspection")

df_reinspect_result = spark.createDataFrame(reinspect_result_rows, schema=inspection_result_schema)
df_reinspect_result.write.format("delta").mode("append").saveAsTable("silver.inspection_result")


print(f"silver.inspection total: {spark.table('silver.inspection').count()} rows")
spark.table("silver.inspection").groupBy("round_no", "overall_result").count().orderBy("round_no").show()

silver.inspection total: 52 rows
+--------+--------------+-----+
|round_no|overall_result|count|
+--------+--------------+-----+
|       1|          FAIL|   24|
|       1|          PASS|    8|
|       2|          PASS|   20|
+--------+--------------+-----+



In [10]:
random.seed(42)

# gap caught mid-build: no one had the CLEANER role yet
df_new_role = spark.createDataFrame(
    [Row(staff_id=3, role_code="CLEANER", valid_from=today.date(), valid_to=None)],
    schema=spark.table("silver.staff_role").schema,
)
df_new_role.write.format("delta").mode("append").saveAsTable("silver.staff_role")

from pyspark.sql import Window
from pyspark.sql import functions as F

latest_inspection = spark.table("silver.inspection") \
    .withColumn("rn", F.row_number().over(Window.partitionBy("vehicle_id").orderBy(F.desc("round_no")))) \
    .filter("rn = 1 AND overall_result = 'PASS'") \
    .select("vehicle_id", "inspection_date")

rows = latest_inspection.collect()

cleaning_schema = StructType([
    StructField("cleaning_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("staff_id", LongType(), True),
    StructField("task_date", DateType(), False),
    StructField("status", StringType(), False),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
])

cleaning_rows = []
cid = 1

for row in rows:
    elapsed = (today_date - row.inspection_date).days
    if elapsed < 1:
        continue

    task_date = row.inspection_date + timedelta(days=1)
    status = "REDO_REQUIRED" if random.random() < 0.05 else "DONE"

    cleaning_rows.append(Row(
        cleaning_id=cid, vehicle_id=row.vehicle_id, staff_id=3,
        task_date=task_date, status=status,
        source_id=1, source_record_id=f"FORM-CLEAN-{cid:04d}",
        created_at=datetime.combine(task_date, datetime.min.time()),
        updated_at=today,
    ))
    cid += 1

df_cleaning = spark.createDataFrame(cleaning_rows, schema=cleaning_schema)
df_cleaning.write.format("delta").mode("overwrite").saveAsTable("silver.cleaning_task")

print(f"silver.cleaning_task: {spark.table('silver.cleaning_task').count()} rows")
df_cleaning.groupBy("status").count().show()

silver.cleaning_task: 28 rows
+-------------+-----+
|       status|count|
+-------------+-----+
|         DONE|   24|
|REDO_REQUIRED|    4|
+-------------+-----+



In [11]:
random.seed(42)
from pyspark.sql import functions as F

FX_RATE_USD_NGN = Decimal("1600.00")  # placeholder rate for this demo
MARKUP_MIN, MARKUP_MAX = 1.25, 1.45
CHANNELS = ["Showroom", "Online", "Social Media"]

cleaned = spark.table("silver.cleaning_task").filter("status = 'DONE'") \
    .select("vehicle_id", "task_date").collect()

purchase_cost = {r.vehicle_id: r.price_amount for r in
                  spark.table("silver.purchase").select("vehicle_id", "price_amount").collect()}

repair_cost = {r.vehicle_id: r.total for r in
               spark.table("silver.repair_job_part")
               .join(spark.table("silver.repair_job"), "repair_job_id")
               .groupBy("vehicle_id")
               .agg(F.sum(F.col("unit_cost") * F.col("qty")).alias("total"))
               .collect()}

listing_schema = StructType([
    StructField("listing_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("price_amount", DecimalType(14, 2), False),
    StructField("currency", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("listed_date", DateType(), False),
    StructField("status", StringType(), False),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

listing_rows = []
lid = 1

for row in cleaned:
    elapsed = (today_date - row.task_date).days
    if elapsed < 1:
        continue

    cost_usd = purchase_cost.get(row.vehicle_id, Decimal("0"))
    total_cost_ngn = (cost_usd * FX_RATE_USD_NGN) + repair_cost.get(row.vehicle_id, Decimal("0"))
    markup = round(random.uniform(MARKUP_MIN, MARKUP_MAX), 2)
    raw_price = total_cost_ngn * Decimal(str(markup))
    price = Decimal(round(float(raw_price) / 5000) * 5000)

    listed_date = row.task_date + timedelta(days=1)

    listing_rows.append(Row(
        listing_id=lid, vehicle_id=row.vehicle_id,
        price_amount=price, currency="NGN",
        channel=random.choice(CHANNELS),
        listed_date=listed_date, status="ACTIVE",
        source_id=1, source_record_id=f"FORM-LIST-{lid:04d}",
        created_at=datetime.combine(listed_date, datetime.min.time()),
        updated_at=today, is_deleted=False, deleted_at=None,
    ))
    lid += 1

df_listing = spark.createDataFrame(listing_rows, schema=listing_schema)
df_listing.write.format("delta").mode("overwrite").saveAsTable("silver.listing")

print(f"silver.listing: {spark.table('silver.listing').count()} rows")
spark.table("silver.listing").select("vehicle_id", "price_amount", "channel").limit(10).toPandas()

silver.listing: 24 rows


,vehicle_id,price_amount,channel
0,35,6690000.00,Showroom
1,38,11350000.00,Social Media
2,39,6135000.00,Showroom
3,9,13755000.00,Showroom
4,10,5710000.00,Showroom
5,11,18065000.00,Online
6,6,7735000.00,Online
7,7,9075000.00,Showroom
8,8,5075000.00,Social Media
9,40,9810000.00,Social Media


In [12]:
from delta.tables import DeltaTable

random.seed(42)
Faker.seed(42)

listings = spark.table("silver.listing").select("listing_id", "vehicle_id", "price_amount", "listed_date").collect()

customer_schema = StructType([
    StructField("customer_id", LongType(), False),
    StructField("name", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("address", StringType(), True),
    StructField("created_at", TimestampType(), False),
])

sale_schema = StructType([
    StructField("sale_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("listing_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("salesperson_staff_id", LongType(), True),
    StructField("agreed_price", DecimalType(14, 2), False),
    StructField("currency", StringType(), False),
    StructField("sale_date", DateType(), False),
    StructField("status", StringType(), False),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
    StructField("is_deleted", BooleanType(), False),
    StructField("deleted_at", TimestampType(), True),
])

payment_schema = StructType([
    StructField("payment_id", LongType(), False),
    StructField("sale_id", LongType(), False),
    StructField("amount", DecimalType(14, 2), False),
    StructField("currency", StringType(), False),
    StructField("method", StringType(), True),
    StructField("payment_date", DateType(), False),
    StructField("created_at", TimestampType(), False),
])

PAYMENT_METHODS = ["BANK_TRANSFER", "POS", "CASH"]

customer_rows, sale_rows, payment_rows, sold_vehicle_ids = [], [], [], []
cust_id, sale_id, pay_id = 1, 1, 1

for row in listings:
    elapsed = (today_date - row.listed_date).days
    time_to_sale = random.randint(5, 60)
    if elapsed < time_to_sale:
        continue

    sale_date = row.listed_date + timedelta(days=time_to_sale)
    discount = round(random.uniform(0, 0.08), 3)
    agreed_price = (row.price_amount * Decimal(str(1 - discount))).quantize(Decimal("1"))

    customer_rows.append(Row(
        customer_id=cust_id, name=fake.name(),
        phone=fake.bothify(text="+234##########"), address=fake.city(),
        created_at=datetime.combine(sale_date, datetime.min.time()),
    ))

    is_installment = random.random() < 0.20
    status = "PENDING_PAYMENT" if is_installment else "FINALIZED"

    sale_rows.append(Row(
        sale_id=sale_id, vehicle_id=row.vehicle_id, listing_id=row.listing_id,
        customer_id=cust_id, salesperson_staff_id=6,
        agreed_price=agreed_price, currency="NGN",
        sale_date=sale_date, status=status,
        source_id=1, source_record_id=f"FORM-SALE-{sale_id:04d}",
        created_at=datetime.combine(sale_date, datetime.min.time()),
        updated_at=today, is_deleted=False, deleted_at=None,
    ))

    pay_amount = (agreed_price * Decimal("0.5")).quantize(Decimal("1")) if is_installment else agreed_price
    payment_rows.append(Row(
        payment_id=pay_id, sale_id=sale_id, amount=pay_amount, currency="NGN",
        method=random.choice(PAYMENT_METHODS), payment_date=sale_date,
        created_at=datetime.combine(sale_date, datetime.min.time()),
    ))
    pay_id += 1

    sold_vehicle_ids.append(row.vehicle_id)
    cust_id += 1
    sale_id += 1

df_customer = spark.createDataFrame(customer_rows, schema=customer_schema)
df_customer.write.format("delta").mode("overwrite").saveAsTable("silver.customer")

df_sale = spark.createDataFrame(sale_rows, schema=sale_schema)
df_sale.write.format("delta").mode("overwrite").saveAsTable("silver.sale")

df_payment = spark.createDataFrame(payment_rows, schema=payment_schema)
df_payment.write.format("delta").mode("overwrite").saveAsTable("silver.payment")

if sold_vehicle_ids:
    listing_table = DeltaTable.forName(spark, "silver.listing")
    listing_table.update(
        condition=f"vehicle_id IN ({','.join(str(v) for v in sold_vehicle_ids)})",
        set={"status": "'SOLD'", "updated_at": "current_timestamp()"}
    )

print(f"silver.customer: {spark.table('silver.customer').count()} rows")
print(f"silver.sale: {spark.table('silver.sale').count()} rows")
df_sale.groupBy("status").count().show()
print(f"silver.payment: {spark.table('silver.payment').count()} rows")
spark.table("silver.listing").groupBy("status").count().show()

silver.customer: 15 rows
silver.sale: 15 rows
+---------------+-----+
|         status|count|
+---------------+-----+
|      FINALIZED|   14|
|PENDING_PAYMENT|    1|
+---------------+-----+

silver.payment: 15 rows
+------+-----+
|status|count|
+------+-----+
|  SOLD|   15|
|ACTIVE|    9|
+------+-----+



In [13]:
from pyspark.sql import functions as F
from collections import defaultdict

def collect_dates(table, date_col, stage, filter_expr=None):
    df = spark.table(table)
    if filter_expr:
        df = df.filter(filter_expr)
    return [(r.vehicle_id, stage, getattr(r, date_col)) for r in df.select("vehicle_id", date_col).collect() if getattr(r, date_col) is not None]

events = []
events += collect_dates("silver.purchase", "purchase_date", "PURCHASED")
events += collect_dates("silver.shipment", "etd", "SHIPPED")
events += collect_dates("silver.port_arrival", "arrival_date", "AT_PORT")
events += collect_dates("silver.port_arrival", "cleared_date", "CLEARED", "clearance_status='CLEARED'")
events += collect_dates("silver.pickup", "pickup_date", "PICKED_UP")
events += collect_dates("silver.office_intake", "intake_date", "AT_OFFICE")
events += collect_dates("silver.inspection", "inspection_date", "INSPECTING", "round_no=1")
events += collect_dates("silver.repair_job", "start_date", "IN_REPAIR")
events += collect_dates("silver.cleaning_task", "task_date", "CLEANED", "status='DONE'")
events += collect_dates("silver.listing", "listed_date", "LISTED")
events += collect_dates("silver.sale", "sale_date", "SOLD")

by_vehicle = defaultdict(list)
for vid, stage, d in events:
    by_vehicle[vid].append((d, stage))

history_schema = StructType([
    StructField("history_id", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("from_stage", StringType(), True),
    StructField("to_stage", StringType(), False),
    StructField("changed_at", TimestampType(), False),
    StructField("source_id", LongType(), False),
    StructField("source_record_id", StringType(), True),
])

history_rows, latest_stage = [], {}
hid = 1
for vid, evs in by_vehicle.items():
    evs.sort(key=lambda x: x[0])
    prev_stage = None
    for d, stage in evs:
        history_rows.append(Row(
            history_id=hid, vehicle_id=vid, from_stage=prev_stage, to_stage=stage,
            changed_at=datetime.combine(d, datetime.min.time()),
            source_id=1, source_record_id=None,
        ))
        hid += 1
        prev_stage = stage
    latest_stage[vid] = prev_stage

df_history = spark.createDataFrame(history_rows, schema=history_schema)
df_history.write.format("delta").mode("overwrite").saveAsTable("silver.vehicle_stage_history")
print(f"silver.vehicle_stage_history: {spark.table('silver.vehicle_stage_history').count()} rows")

# real MERGE, same pattern as Phase 6 -- one set-based update, not a loop
updates = [Row(vehicle_id=vid, current_stage=stage, current_status=("SOLD" if stage == "SOLD" else "ACTIVE"))
           for vid, stage in latest_stage.items()]
df_updates = spark.createDataFrame(updates)

vehicle_table = DeltaTable.forName(spark, "silver.vehicle")
(vehicle_table.alias("v")
 .merge(df_updates.alias("u"), "v.vehicle_id = u.vehicle_id")
 .whenMatchedUpdate(set={
     "current_stage": "u.current_stage",
     "current_status": "u.current_status",
     "updated_at": "current_timestamp()",
 })
 .execute())

print("vehicle.current_stage after merge:")
spark.table("silver.vehicle").groupBy("current_stage", "current_status").count().orderBy(F.desc("count")).show()

silver.vehicle_stage_history: 330 rows
vehicle.current_stage after merge:
+-------------+--------------+-----+
|current_stage|current_status|count|
+-------------+--------------+-----+
|         SOLD|          SOLD|   15|
|       LISTED|        ACTIVE|    9|
|    IN_REPAIR|        ACTIVE|    6|
|      AT_PORT|        ACTIVE|    4|
|      SHIPPED|        ACTIVE|    4|
|   INSPECTING|        ACTIVE|    2|
+-------------+--------------+-----+

